In [3]:
import pandas as pd
import joblib

X_train = pd.read_parquet("../data/processed/X_train.parquet")
X_test = pd.read_parquet("../data/processed/X_test.parquet")
y_train = pd.read_parquet("../data/processed/y_train.parquet")['Churn']
y_test = pd.read_parquet("../data/processed/y_test.parquet")['Churn']
preprocessor = joblib.load("../data/processed/preprocessor.joblib")

## Model Comparison - LogisticRegression vs RandomForest Classifier

Baseline comparison of both models ('class_weight='balanced', from previous decision) using 5-folds `StratifiedKFold` cross-validation on the training set. F1 and recall (minority class) are tracked

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

logreg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000))
])

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42))
])

In [5]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

logreg_scores = cross_validate(logreg_pipeline, X_train, y_train, cv=cv, scoring=['accuracy', 'precision', 'recall', 'f1'], return_train_score=True)

rf_scores = cross_validate(rf_pipeline, X_train, y_train, cv=cv, scoring=['accuracy', 'precision', 'recall', 'f1'], return_train_score=True)

In [6]:
comparison_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [logreg_scores['test_accuracy'].mean(), rf_scores['test_accuracy'].mean()],
    'Precision': [logreg_scores['test_precision'].mean(), rf_scores['test_precision'].mean()],
    'Recall': [logreg_scores['test_recall'].mean(), rf_scores['test_recall'].mean()],
    'F1-Score': [logreg_scores['test_f1'].mean(), rf_scores['test_f1'].mean()]
})

print(comparison_df)

                 Model  Accuracy  Precision    Recall  F1-Score
0  Logistic Regression  0.749914   0.518611  0.803344  0.630239
1        Random Forest  0.790916   0.646413  0.469565  0.543921


**Findings:** LogisticRegression catches 80% of churners; RandomForest catches 47%. That's the number that decides this, a missed churner causes no chance of intervention, so recall matters more so than accuracy or precision RandomForest wins on (79.1% vs 75.0% accuracy, 0.646 vs 0.519 precision). F1 actually favors RandomForest (0.544 vs 0.630 for LogReg), but F1 is the wrong lens here as it rewards a precision/recall balance this problem doesn't need.

The gap probably comes down to how `class_weight='balanced'` hits each model differently. LogisticRegression reweights the loss directly, so it's a strong, blunt push toward flagging minority-class rows. RandomForest reweights impurity at each split instead, and that effect thins out once you're averaging votes across 100 trees. Neither model is tuned yet, so this isn't the two algorithms at their best but at default setting up

**Decision:** LogisticRegression is the model going forward. RandomForest stays on the
bench. If tuning LogReg stalls, or if false positives start costing more than assumed,
it's the first thing to revisit.